# Unsupervised Ellipsoid Fitting Algorithm

This notebook extends the hypersphere-based algorithm by replacing each spherical region with an ellipsoid. The motivation is that normal embeddings in DINOv2 feature space are unlikely to form locally isotropic clusters. Instead, neighbourhoods may stretch more strongly along some directions than others. Ellipsoids are therefore able to model local variance more naturally than hyperspheres.

The ellipsoid formulation has several advantages:

- Aligns each region with the natural variance structure of the local KNN neighbourhood.
- Reduces unused empty space compared with hyperspheres, since the boundary can contract along low-variance directions.
- It can reduce unnecessary overlap between neighbouring regions by following the dominant principal axes of the local embedding distribution.
- It is better suited to high-variance categories, where the normal embedding space may contain elongated or anisotropic regions.
- Provides additional interpretability through eigenvalues, eigenvectors, axis ratios, and local region structure.

Several changes were introduced compared with the hypersphere version:

- Growth is variance-scaled rather than uniform. Expansion along each axis is controlled by the relative eigenvalue contribution, so high-variance directions can grow more than low-variance directions.
- Candidate cleaning is weight-based. Instead of immediately removing a point when a candidate ellipsoid overlaps a previous region, the algorithm first reduces that point’s contribution to the ellipsoid  fit. If its weight reaches zero and overlap remains, the point is removed.
- Sparse ellipsoids require additional support. Unlike hyperspheres, ellipsoids fitted from very few points can become geometrically unstable. To address this, the covariance of a small candidate region is blended with covariance information from a previous ellipsoid.
- The current support strategy borrows covariance from the nearest ellipsoid by centre distance. This is a limitation, since the nearest ellipsoid may not be the most geometrically similar. Future work should select support using both spatial proximity and shape similarity.

In [7]:
import os

import torch
import sqlite3
import pandas as pd

import json

from datetime import datetime
from dataclasses import asdict

from pathlib import Path
import sys 

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, EXPERIMENTS, RESULTS, DB_PATH, MODELS
MODEL_PATH = MODELS / "dino_pretrained" #"dino_adapter_block/20260812_150510"

EMBED_PATH = EMBEDS_DIR / "dino/pretrained" #"dino/20260812_150510"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = EXPERIMENTS / EMBED_NAME / "ellipsoid"
RESULTS_DIR = RESULTS / EMBED_NAME

In [8]:
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [9]:
metadata_path = MODEL_PATH / "metadata.json"

if not metadata_path.exists():
    with open(metadata_path, "w") as f:
        json.dump({}, f)

In [10]:
cls_tokens = torch.load(EMBED_PATH/"cls.pt", weights_only=False)

In [11]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [12]:
%load_ext autoreload
%autoreload 2

from src.algorithims.ellipsoid import EllipsoidFitter, EllipsoidCover, CandidateCleaner, EllipsoidEvaluator
from src.types import ExperimentConfig, AlgorithmResults

metadata = ExperimentConfig(
    K_frac=0.05,
    start_growth=1.2,
    min_growth=1,
    reg=1e-4,
    growth_type="variance_scaled",
    cleaner="shared_axis"
)

fitter = EllipsoidFitter(support_points=5, reg=metadata.reg)
cleaner = CandidateCleaner(min_points=1, fitter=fitter)

cover = EllipsoidCover(fitter=fitter, cleaner=cleaner)

evaluator = EllipsoidEvaluator(reg=metadata.reg)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
with open(MODEL_PATH / "metadata.json", "r") as f:
    model_metadata = json.load(f)
    
neg_indices = model_metadata.get("negative_indices", [])

print(model_metadata)
print(model_metadata.keys())

{}
dict_keys([])


In [14]:
time = datetime.now().strftime("%y-%m-%d_%H-%M-%S")
aurocs = {}

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category / time
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[
        (~train_mask)
        & (~meta.index.isin(neg_indices))
    ]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    collection = cover.run(
        embeds=cat_emb, output_dir=outputs_dir, 
        k_frac=metadata.K_frac, 
        start_growth=metadata.start_growth, min_growth=metadata.min_growth
        )
    
    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[
        (~train_mask)
        & (~meta.index.isin(neg_indices))
    ]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

    overlaps_df, num_overlaps = evaluator.overlap(embeds=cat_emb, ellipsoids=collection.ellipsoids)
    overlaps_df.to_csv(outputs_dir / f"overlaps.csv", index=False)

    good_any, good_counts = evaluator.inside_any_count(good_test_emb, collection.ellipsoids)
    defect_any, defect_counts = evaluator.inside_any_count(defect_test_emb, collection.ellipsoids)

    evaluation = evaluator.evaluate_detection(good_test_emb, defect_test_emb, collection)
    evaluation.samples.to_csv(outputs_dir / f"results.csv", index=False)

    diagnostics = evaluator.bucket_diagnostics(evaluation.samples)

    aurocs[category] = evaluation.metrics.auroc

    results = AlgorithmResults(
        config=metadata,
        category=category,
        n_shapes=len(collection),
        auroc=evaluation.metrics.auroc,
        normal_inside=int(good_any.sum()),
        defect_inside=int(defect_any.sum())
    )

    with open(outputs_dir / f"metadata.json", "w") as f:
        json.dump(asdict(results), f)

aurocs_df = pd.DataFrame(aurocs.items(), columns=["Category", "AUROC"])
aurocs_df.round(3).to_csv(RESULTS_DIR / "ellip_auroc.csv", index=False)

Running bottle
Running cable
Running capsule
Running carpet
Running grid
Running hazelnut
Running leather
Running metal_nut
Running pill
Running screw
Running tile
Running toothbrush
Running transistor
Running wood
Running zipper


In [15]:
aurocs_df

,Category,AUROC
0,bottle,0.996825
1,cable,0.882684
2,capsule,0.874352
3,carpet,0.972311
4,grid,0.969925
5,hazelnut,0.930714
6,leather,0.999660
7,metal_nut,0.934995
8,pill,0.910256
9,screw,0.807747


In [16]:
df_roc_stats = pd.DataFrame({
    "mean": aurocs_df["AUROC"].mean(),
    "median": aurocs_df["AUROC"].median(),
    "std": aurocs_df["AUROC"].std(),
    "min_cat":  aurocs_df["Category"][aurocs_df["AUROC"].idxmin()],
    "min": aurocs_df["AUROC"].min(),
    "max_cat": aurocs_df["Category"][aurocs_df["AUROC"].idxmax()],
    "max": aurocs_df["AUROC"].max()
}, index=[0]).round(3)

df_roc_stats.to_csv(RESULTS_DIR / "ellipsoid_auroc_stats.csv", index=False)

aurocs_df = aurocs_df.round(3)
aurocs_df.to_csv(RESULTS_DIR /"ellipsoid_aurocs.csv", index=False)

df_roc_stats

,mean,median,std,min_cat,min,max_cat,max
0,0.934,0.935,0.056,screw,0.808,leather,1.0


In [17]:
display(diagnostics["n_points"])
display(diagnostics["eig_ratio"])
display(diagnostics["n_points_by_class"])
display(diagnostics["eig_ratio_aurocs"])

,mean,count
n_points_bucket,,
"(-0.001, 3.0]",0.90,100
"(3.0, 5.0]",0.88,25
"(5.0, 10.0]",1.00,4
"(10.0, 100.0]",1.00,22


,mean,count
eigval_ratio_bucket,,
"(6.02, 318.467]",0.947368,38
"(318.467, 2345.004]",0.829787,47
"(2345.004, 4550.72]",0.945455,55
"(4550.72, 21119.899]",1.000000,11


mean  count
n_points_bucket y_true                 
(-0.001, 3.0]   0.0     1.000000     10
                1.0     0.888889     90
(3.0, 5.0]      0.0     1.000000      5
                1.0     0.850000     20
(5.0, 10.0]     0.0     1.000000      3
                1.0     1.000000      1
(10.0, 100.0]   0.0     1.000000     14
                1.0     1.000000      8

,eigval_ratio_bucket,auroc,count
0,"(6.02, 318.467]",0.997222,38
1,"(318.467, 2345.004]",0.923810,47
2,"(2345.004, 4550.72]",0.979167,55
3,"(4550.72, 21119.899]",1.000000,11


In [18]:
ellipsoids_df = collection.to_dataframe()

ellipsoids_df

,ellipsoid_id,raw_eig_ratio,reg_eig_ratio,pc95,rank,pc1_ratio,n_points,threshold,support_id,weights_mean,weights_min,n_reduced_weights
0,0,7.502700,7.502112,11,12,0.160215,13,11.076678,NaN,1.0,1.0,0
1,1,7.716991,7.716535,10,11,0.207949,12,10.083107,NaN,1.0,1.0,0
2,2,7.484683,7.484287,9,10,0.220525,11,9.090737,NaN,1.0,1.0,0
3,3,7.401708,7.401338,9,10,0.223016,11,9.090739,NaN,1.0,1.0,0
4,4,10.017386,10.016851,8,9,0.283030,10,8.099868,NaN,1.0,1.0,0
5,5,7.914805,7.914421,8,9,0.241227,10,8.099898,NaN,1.0,1.0,0
6,6,6.977432,6.977169,7,8,0.263981,9,7.111024,NaN,1.0,1.0,0
7,7,6.021317,6.021112,7,8,0.264324,9,7.111020,NaN,1.0,1.0,0
8,8,5.489769,5.489620,6,7,0.250504,8,6.124939,NaN,1.0,1.0,0
9,9,8.999574,8.999275,6,7,0.372675,8,6.124948,NaN,1.0,1.0,0


In [19]:
print(ellipsoids_df["pc1_ratio"].mean())
print(ellipsoids_df["pc1_ratio"].median())
print(ellipsoids_df["pc95"].mean())
print(ellipsoids_df["pc95"].median())

0.39183233016038294
0.40265676207551393
6.777777777777778
6.0


In [20]:
metrics_df = pd.DataFrame([evaluation.metrics.to_dict()])

metrics_df

,auroc,best_threshold,accuracy,good_accuracy,defect_accuracy,mean_good_score,mean_defect_score,score_gap,false_pos,false_neg,n_winning_ellipsoid,max_winning_fraction
0,0.982668,898703.963557,0.913907,1.0,0.890756,562284.159877,1.915521e+06,1.353236e+06,0,13,21,0.284768


In [21]:
summary = []

for cat in categories:
    cat_dir = EXPERIMENTS_DIR / cat

    # Find latest run folder
    runs = sorted(
        [p for p in cat_dir.iterdir() if p.is_dir()],
        reverse=True
    )

    if not runs:
        continue

    results_path = runs[0] / "results.csv"

    if not results_path.exists():
        continue

    s = pd.read_csv(results_path)

    n_normal_inside = (
        (s["y_true"] == 0) &
        (s["score"] < 0)
    ).sum()

    n_defect_outside = (
        (s["y_true"] == 1) &
        (s["score"] > 0)
    ).sum()

    summary.append({
        "category": cat,
        "normal_inside": n_normal_inside,
        "defect_outside": n_defect_outside
    })

summary_df = pd.DataFrame(summary)
display(summary_df.sort_values(
    ["normal_inside", "defect_outside"],
    ascending=False
))

,category,normal_inside,defect_outside
8,pill,0,141
9,screw,0,119
14,zipper,0,119
2,capsule,0,109
7,metal_nut,0,93
1,cable,0,92
6,leather,0,92
3,carpet,0,89
10,tile,0,84
5,hazelnut,0,70


In [22]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from matplotlib.patches import Ellipse
from src.algorithims.ellipsoid.distance import distance_squared


def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


# -------------------------------------------------------
# 1. Pick a defective sample outside all learned regions
# -------------------------------------------------------

samples = evaluation.samples.copy()
n_good = len(good_test_emb)

defect_rows = samples[
    (samples["y_true"] == 1) &
    (samples["score"] > 0)
].sort_values("score")

print("Number of defects outside all regions:", len(defect_rows))

# Use roughly the median outside defect.
# This avoids choosing either a barely-outside case or some extreme outlier.
row_idx = defect_rows.index[len(defect_rows) // 2]

defect_idx = row_idx - n_good

x = to_numpy(defect_test_emb[defect_idx])

score = samples.loc[row_idx, "score"]
winning_idx = int(samples.loc[row_idx, "winning_ellipsoid"])
winning = collection.ellipsoids[winning_idx]

print("Selected sample:")
print(samples.loc[row_idx])

Number of defects outside all regions: 119
Selected sample:
y_true                             1.0
predicted_label                      1
score                   1519268.406175
correct                           True
winning_ellipsoid                   23
winning_n_points                     4
winning_eigval_ratio        534.034402
Name: 89, dtype: object


In [23]:
# -------------------------------------------------------
# 2. PCA for visualisation only
# -------------------------------------------------------

train_np = to_numpy(cat_emb)

pca = PCA(n_components=2)
train_2d = pca.fit_transform(train_np)

sample_2d = pca.transform(x[None, :])[0]

centres = np.vstack([
    to_numpy(e.center)
    for e in collection.ellipsoids
])

centres_2d = pca.transform(centres)

winning_center_2d = centres_2d[winning_idx]

print(
    "PCA variance explained:",
    pca.explained_variance_ratio_.sum()
)

PCA variance explained: 0.30384928


In [24]:
def projected_ellipse(ellipsoid, pca, reg=1e-4):
    V = to_numpy(ellipsoid.eigvecs)
    lambdas = to_numpy(ellipsoid.eigvals)

    d = V.shape[0]

    covariance = (
        V @ np.diag(lambdas) @ V.T
        + reg * np.eye(d)
    )

    A = pca.components_
    covariance_2d = A @ covariance @ A.T

    vals, vecs = np.linalg.eigh(covariance_2d)

    order = np.argsort(vals)[::-1]
    vals = vals[order]
    vecs = vecs[:, order]

    axes = np.sqrt(
        np.maximum(vals, 0) * ellipsoid.threshold
    )

    width = 2 * axes[0]
    height = 2 * axes[1]

    angle = np.degrees(
        np.arctan2(vecs[1, 0], vecs[0, 0])
    )

    return width, height, angle

def sci_notation(value, decimals=2):
    if value == 0:
        return "0"

    exponent = int(np.floor(np.log10(abs(value))))
    coefficient = value / (10 ** exponent)

    return rf"{coefficient:.{decimals}f} $\times 10^{{{exponent}}}$"

def region_metrics(x, ellipsoid, reg=1e-4):
    diff = to_numpy(x)[None, :] - ellipsoid.center

    d2 = distance_squared(
        diff,
        ellipsoid.eigvecs,
        ellipsoid.eigvals,
        reg=reg
    )[0]

    raw_margin = d2 - ellipsoid.threshold
    boundary_ratio = d2 / ellipsoid.threshold
    relative_margin = boundary_ratio - 1

    return raw_margin, boundary_ratio, relative_margin

In [25]:
# -------------------------------------------------------
# Plot defect against its 3 closest ellipsoids
# -------------------------------------------------------

from matplotlib.patches import Ellipse

# Actual model scoring margin against every ellipsoid
all_margins = get_margins(
    x,
    collection.ellipsoids,
    reg=metadata.reg
)

# IMPORTANT:
# Winning/nearest regions are still selected using the
# original model score, not the normalised boundary ratio.
nearest_indices = np.argsort(all_margins)[:3]

print("Closest ellipsoids:")
for i in nearest_indices:
    margin, ratio, relative = region_metrics(
        x,
        collection.ellipsoids[i],
        metadata.reg
    )

    print(
        f"E{i}: "
        f"margin={margin:.4e}, "
        f"boundary ratio={ratio:.2f}x, "
        f"relative margin={relative:.2f}"
    )


fig, ax = plt.subplots(figsize=(9, 7))

# Normal training embeddings
ax.scatter(
    train_2d[:, 0],
    train_2d[:, 1],
    s=20,
    alpha=0.25,
    label="Normal training embeddings"
)

# All learned ellipsoid centres
ax.scatter(
    centres_2d[:, 0],
    centres_2d[:, 1],
    marker="x",
    s=35,
    alpha=0.25,
    label="Other ellipsoid centres"
)


# -------------------------------------------------------
# Draw the three closest regions
# -------------------------------------------------------

for rank, idx in enumerate(nearest_indices):

    ellipsoid = collection.ellipsoids[idx]
    centre_2d = centres_2d[idx]

    width, height, angle = projected_ellipse(
        ellipsoid,
        pca,
        reg=metadata.reg
    )

    is_winner = rank == 0

    ellipse_patch = Ellipse(
        xy=centre_2d,
        width=width,
        height=height,
        angle=angle,
        fill=False,
        linewidth=2.5 if is_winner else 1.5,
        linestyle="-" if is_winner else "--",
        alpha=1.0 if is_winner else 0.6,
        label=(
            f"Winning region E{idx}"
            if is_winner
            else (
                "Alternative candidate regions"
                if rank == 1
                else None
            )
        )
    )

    ax.add_patch(ellipse_patch)

    # Highlight region centre
    ax.scatter(
        centre_2d[0],
        centre_2d[1],
        marker="X",
        s=130 if is_winner else 80,
        alpha=1.0 if is_winner else 0.6,
        zorder=5
    )

    # Visual connection only.
    # The line itself is NOT the model's distance calculation.
    ax.plot(
        [sample_2d[0], centre_2d[0]],
        [sample_2d[1], centre_2d[1]],
        linewidth=1.7 if is_winner else 1.0,
        linestyle="--",
        alpha=1.0 if is_winner else 0.5
    )

    # Get both actual margin and easier-to-read boundary ratio
    margin, ratio, relative = region_metrics(
        x,
        ellipsoid,
        metadata.reg
    )

    ax.annotate(
        f"E{idx}\n"
        f"{ratio:.2f}x boundary",
        centre_2d,
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=9
    )


# -------------------------------------------------------
# Defective sample
# -------------------------------------------------------

ax.scatter(
    sample_2d[0],
    sample_2d[1],
    marker="*",
    s=280,
    zorder=6,
    label="Defective test sample"
)

# Winning region's explanatory boundary ratio
winning_ellipsoid = collection.ellipsoids[nearest_indices[0]]

winning_margin, winning_ratio, winning_relative = region_metrics(
    x,
    winning_ellipsoid,
    metadata.reg
)

ax.annotate(
    f"Defect\n"
    f"Closest region: E{nearest_indices[0]}\n"
    f"{winning_ratio:.2f}x boundary",
    sample_2d,
    xytext=(10, -48),
    textcoords="offset points",
    fontsize=10
)


# -------------------------------------------------------
# Labels
# -------------------------------------------------------

ax.set_title(
    "Interpreting an anomalous sample using Cloud of Ellipsoids\n"
    f"MVTec AD - {category}"
)

ax.set_xlabel("PCA component 1")
ax.set_ylabel("PCA component 2")

ax.legend()

fig.text(
    0.5,
    0.01,
    "The sample is scored against every learned ellipsoid; the minimum raw boundary margin\n"
    "determines the winning region and anomaly score. Boundary ratios are shown for interpretability.\n"
    "PCA is used only for visualisation; scoring is performed in the original 384-dimensional space.",
    ha="center",
    fontsize=8
)

plt.tight_layout(rect=[0, 0.08, 1, 1])

plt.savefig(
    "cloud_of_ellipsoids_interpretability.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

NameError: name 'get_margins' is not defined